NAME: PRIYANKA RAJESH GADRE

INTERN ID: IN226049302

GenAI Task 2: NLP

In [ ]:
# ==============================
# 1. Import Libraries
# ==============================
import numpy as np
import pandas as pd
import re
import string

In [ ]:
# NLP Libraries
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize


In [ ]:
# ML Libraries
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [ ]:
# Download required resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [28]:
# ==============================
# 2. Load Dataset (Working Link)
# ==============================
url = "https://raw.githubusercontent.com/laxmimerit/All-CSV-ML-Data-Files-Download/master/IMDB-Dataset.csv"
df = pd.read_csv(url)

print("Dataset Shape:", df.shape)
print("\nSample Data:\n", df.head())

Dataset Shape: (50000, 2)

Sample Data:
                                               review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [30]:

# 3. Data Understanding
# ==============================
print("\nClass Distribution:\n", df['sentiment'].value_counts())

# Convert labels
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})


Class Distribution:
 sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [31]:
# ==============================
# 4. NLP Preprocessing
# ==============================
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)

    tokens = text.split()   # safer than word_tokenize

    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return " ".join(tokens)

# Apply preprocessing
df['clean_text'] = df['review'].apply(clean_text)

print("\nSample Cleaned Data:\n", df[['review','clean_text']].head())


Sample Cleaned Data:
                                               review  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br /><br />The...   
2  I thought this was a wonderful way to spend ti...   
3  Basically there's a family where a little boy ...   
4  Petter Mattei's "Love in the Time of Money" is...   

                                          clean_text  
0  one reviewer mentioned watching oz episode you...  
1  wonderful little production filming technique ...  
2  thought wonderful way spend time hot summer we...  
3  basically there family little boy jake think t...  
4  petter matteis love time money visually stunni...  


In [32]:
# ==============================
# 5. Train-Test Split
# ==============================
X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [33]:
# ==============================
# 6. Feature Engineering
# ==============================
# Bag of Words
bow = CountVectorizer(max_features=5000)
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

# TF-IDF
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [38]:
# ==============================
# 7. Model Evaluation Function
# ==============================
results = []

def evaluate_model(model, X_train, X_test, y_train, y_test, method):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    results.append([model.__class__.__name__, method, acc, prec, rec, f1])

    # ✅ PRINT MUST BE INSIDE FUNCTION
    print("\n====================================")
    print("Model:", model.__class__.__name__)
    print("Method:", method)
    print("Accuracy:", acc)
    print("Precision:", prec)
    print("Recall:", rec)
    print("F1 Score:", f1)
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [40]:
# ==============================
# 8. Train Models (BoW)
# ==============================
print("\n=========== USING BAG OF WORDS ===========")

evaluate_model(LogisticRegression(max_iter=200), X_train_bow, X_test_bow, y_train, y_test, "BoW")
evaluate_model(MultinomialNB(), X_train_bow, X_test_bow, y_train, y_test, "BoW")
evaluate_model(DecisionTreeClassifier(), X_train_bow, X_test_bow, y_train, y_test, "BoW")


=========== USING BAG OF WORDS ===========

Model: LogisticRegression
Method: BoW
Accuracy: 0.8696
Precision: 0.8696191610573093
Recall: 0.8696
F1 Score: 0.8695983099940975

Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.87      0.87      5000
           1       0.87      0.87      0.87      5000

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000


Model: MultinomialNB
Method: BoW
Accuracy: 0.8429
Precision: 0.842914937374672
Recall: 0.8429
F1 Score: 0.842898289162369

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.85      0.84      5000
           1       0.85      0.84      0.84      5000

    accuracy                           0.84     10000
   macro avg       0.84      0.84      0.84     10000
weighted avg       0.84      0.84      0.84     10000


In [41]:
# ==============================
# 9. Train Models (TF-IDF)
# ==============================
print("\n=========== USING TF-IDF ===========")

evaluate_model(LogisticRegression(max_iter=200), X_train_tfidf, X_test_tfidf, y_train, y_test, "TF-IDF")
evaluate_model(MultinomialNB(), X_train_tfidf, X_test_tfidf, y_train, y_test, "TF-IDF")
evaluate_model(DecisionTreeClassifier(), X_train_tfidf, X_test_tfidf, y_train, y_test, "TF-IDF")


=========== USING TF-IDF ===========

Model: LogisticRegression
Method: TF-IDF
Accuracy: 0.8858
Precision: 0.885986817619728
Recall: 0.8858
F1 Score: 0.8857861801277955

Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.87      0.88      5000
           1       0.88      0.90      0.89      5000

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000


Model: MultinomialNB
Method: TF-IDF
Accuracy: 0.8526
Precision: 0.852669123148137
Recall: 0.8526
F1 Score: 0.8525927770460752

Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.85      0.85      5000
           1       0.85      0.86      0.85      5000

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000


In [42]:
# ==============================
# 10. Model Comparison
# ==============================
results_df = pd.DataFrame(results, columns=[
    "Model", "Method", "Accuracy", "Precision", "Recall", "F1 Score"
])

print("\n=========== MODEL COMPARISON ===========")
print(results_df.sort_values(by="Accuracy", ascending=False))



=========== MODEL COMPARISON ===========
                    Model  Method  Accuracy  Precision  Recall  F1 Score
4      LogisticRegression  TF-IDF    0.8858   0.885987  0.8858  0.885786
1      LogisticRegression     BoW    0.8696   0.869619  0.8696  0.869598
0      LogisticRegression     BoW    0.8696   0.869619  0.8696  0.869598
5           MultinomialNB  TF-IDF    0.8526   0.852669  0.8526  0.852593
2           MultinomialNB     BoW    0.8429   0.842915  0.8429  0.842898
3  DecisionTreeClassifier     BoW    0.7220   0.722054  0.7220  0.721983
6  DecisionTreeClassifier  TF-IDF    0.7207   0.720700  0.7207  0.720700


In [43]:
# ==============================
# 11. Final Insights
# ==============================
print("\n================ FINAL INSIGHTS ================")
print("""
1. TF-IDF performs better than Bag of Words.
2. Logistic Regression gives best accuracy.
3. Naive Bayes is fast and efficient.
4. Decision Tree may overfit.
5. Preprocessing improves performance significantly.
6. Best combination: TF-IDF + Logistic Regression.
""")


================ FINAL INSIGHTS ================

1. TF-IDF performs better than Bag of Words.
2. Logistic Regression gives best accuracy.
3. Naive Bayes is fast and efficient.
4. Decision Tree may overfit.
5. Preprocessing improves performance significantly.
6. Best combination: TF-IDF + Logistic Regression.

